In [ ]:
import random
import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import os.path
from mpl_toolkits import mplot3d
from matplotlib import colors
import matplotlib.cm as cm
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import math
import sys
import copy
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
#from mpl_toolkits.mplot3d import Axes3D 

Overview


# User Input Desired Settings
Before running code blocks below, create a `Mg22_data` folder and put it in the `voxel_data` folder. Put the `output_digi_HDF_Mg22_Ne20pp_8MeV.h5` file (get that file from Raghu or Michelle) into that folder.

*Note: Data files shouldn't be committed to the Github*


In [ ]:
# change file directory to the .h5 data file to convert
#VOXEL_PATH = "Mg22_pretrain/voxel_data/"
#DATASET = "Mg22_data/"

DATA_PATH = "data/"
os.makedirs(DATA_PATH, exist_ok=True)

`.h5` data files use a Hierarchical Data Format (HDF), which follows a tree-like structure. Each of the highest most nodes are called keys, and these are different categories for the data. In this case, the `output_digi_HDF_Mg22_Ne20pp_8MeV.h5` file is the point cloud file, containing all of the point clouds for each of the events. `original_keys` is the lsit of different events. This data files has 10000 events.

In [ ]:
file = h5py.File(f'{DATA_PATH}output_digi_HDF_Mg22_Ne20pp_8MeV.h5', 'r')

original_keys = list(file.keys())
original_length = len(original_keys)
# print(original_keys)

In [ ]:
#making an array of the lengths of events
event_lens = np.zeros(original_length, int)
for i in range(original_length):
    event = original_keys[i]
    event_lens[i] = len(file[event])

To do our machine learning task, we need to either down- or up- sample the number points in each event to keep the number of points per event consistent. `sample_size` is the variable that sets the specific number of points for each event. It's set at 512 right now, but can be changed if necessary.

In [ ]:
sample_size = 512 #the number of data points in each event 

# x[0], y[1], z[2], time[3], Amplitude[4], trackID (particle ID)[5], pointID[6]
# energy[7] ,energy loss[8] ,angle[9], Mass[10], Atomic number[11], Event_id index[12], number of tracks[13]

# For now, we don't have number of tracks[13] yet

PROJECTION = 'XYZQ' # the three dimensions which will be input into the model
ISOTOPE = 'Mg22'
PROJ_TO_COLS = [0,1,2,4,5,12,13]

PROJ_TO_COLS = {'XYZQ' : [0,1,2,4,5,12,13], 'XYZ': [0,1,2,5,12,13], 'XYQ': [0,1,4,5,12,13], 'XZQ': [0,2,4,5,12,13], 'YZQ': [1,2,4,5,12,13]}

user_input = PROJ_TO_COLS[PROJECTION]
print(user_input)

# Convert Raw H5 File into npArray with Corresponding key index

In [ ]:
#making a numpy array of the data. three dimension are [event number, point within event, data value at point]
#length of each event is based on the longest event in dataset, so non-maximal events are padded with zeros at the end
#12th index of each data point now corresponds to the index of the event in the h5 file's original_keys
# each point thus contains:
# x,y,z, time, Amplitude, trackID (particle ID), pointID, energy, energy loss, angle, Mass, Atomic number, Event_id index
file_name = ISOTOPE + '_w_key_index'
# **only doing this if the file doens't exist already, as the conversion takes a while**
if not os.path.exists(DATA_PATH + file_name + '.npy'):
    event_data = np.zeros((original_length, np.max(event_lens), 13), float) 
    for n in range(len(original_keys)):
        name = original_keys[n]
        event = file[name]
        ev_len = len(event)
        #converting event into an array
        for i,e in enumerate(event):
            instant = np.array(list(e))
            event_data[n][i][:12] = np.array(instant)
            event_data[n][i][-1] = float(n) #insert index value to find corresponding event ID
    np.save(DATA_PATH + file_name, event_data)

### Assertion Statements to Check the Conversion

In [ ]:
data = np.load( DATA_PATH + ISOTOPE + '_w_key_index' + '.npy')
assert data.shape == (original_length, np.max(event_lens), 13), 'Array has incorrect shape'
assert len(np.unique(data[:,:,12])) == original_length, 'Array has incorrect Event_ids'

# Random sample From New Numpy Array

In [ ]:
#adding 13th column to correspond to the number of tracks in event, zero-indexed: 0 = beam, 1= two track, 2 = 3 track...
data_array = ISOTOPE + '_w_key_index.npy' #insert desired array to sample from 
new_array_name = ISOTOPE + '_size' + str(sample_size) + '_sampled'
data = np.load(DATA_PATH + data_array)
new_data = np.zeros((original_length, sample_size, 14), float) 
for i in range(original_length):
    ev_len = event_lens[i]    #length of event-- i.e. number of points
    particle_ids = data[i][:ev_len,5]
    label, distr = np.unique(particle_ids, return_counts=True)
    shortest = label[np.argmin(distr)]
    shortest_ind = np.argwhere(particle_ids == shortest)
    if ev_len == sample_size:    #if array is already preferred length
        new_data[i][:,:-1] = data[i][:ev_len,:]
    else:
        instant = 0
        #the first instances sampled will be those belonging to the shortest track to prevent it from being lost
        for n in range(shortest_ind.size):    
            new_data[i,instant,:-1] = data[i,shortest_ind[n],:]
            instant += 1
        need = sample_size - shortest_ind.size
        random_points = np.random.choice(range(ev_len), need, replace= True if need > ev_len else False)  #choosing random instances to sample
        for r in random_points:
            new_data[i,instant,:-1] = data[i,r,:] 
            instant += 1
    unique_point_ids = np.unique(data[i,:ev_len,5])    #array of unique particle IDs
    new_data[i][0][-1] = unique_point_ids.size - 1    #number of unique particles, scaled to start at 0
np.save(DATA_PATH + new_array_name, new_data) 

### Assertion Statements to Check the Data After Random Sampling

In [ ]:
data = np.load( DATA_PATH + ISOTOPE + '_size' + str(sample_size) + '_sampled.npy')
assert data.shape == (original_length, sample_size, 14), 'Array has incorrect shape'
assert len(np.unique(data[:,:,13])) == len(np.unique(data[:,:,5]))-1, 'Array has incorrect number of tracks'

### Check Distribution of labels after sampling (CHECK THIS LATER!!!!)

In [ ]:
#cheking how the distribution of labels changes from sampling
name = ISOTOPE + '_size' + str(sample_size) + '_sampled'
data = np.load(DATA_PATH + name + '.npy')
real_tracks = np.zeros(original_length,int) 
sampled_tracks = np.zeros(original_length,int)

for i in range(original_length):
    ev_nt = data[i]
    real_tracks[i] = ev_nt[0,-1]
    unique_point_ids = np.unique(ev_nt[:,5])    #array of unqiue particles IDs
    sampled_tracks[i] = unique_point_ids.size - 1
    
og_label, og_distr = np.unique(real_tracks, return_counts=True)
new_label, new_distr = np.unique(sampled_tracks, return_counts=True)
print(og_label, new_label)
print(og_distr)
print(new_distr)
# number of events that have lost a track from sampling. not a big deal if nonzero
print('Events changed = ' + str(np.sum(np.abs(new_distr - og_distr))//2)) 

### Check Distribution of Event Types After Filtering Tracks

In [ ]:
name = ISOTOPE + '_size' + str(sample_size) + '_sampled'
data = np.load(DATA_PATH + name + '.npy')
real_tracks = np.zeros(len(data),int)
sampled_tracks = np.zeros(len(data),int)


for i in range(len(data)):
    ev_nt = data[i]
    real_tracks[i] = ev_nt[0,-1] # original event label
    unique_point_ids = np.unique(ev_nt[:,5])    # array of unqiue particles IDs-- i.e current event label after sampling
    sampled_tracks[i] = unique_point_ids.size - 1
    
label, og_distr = np.unique(real_tracks, return_counts=True)
label, new_distr = np.unique(sampled_tracks, return_counts=True)

print(list(np.unique(data[:,:,5]))) # Particles present in data
print(list(np.unique(data[:,:,13]))) # types of events present in data
print(og_distr) 
print(new_distr) 
unique_point_id = np.unique(data[:,:,5])
print((unique_point_id)) # uncomment to check particle ids
assert (np.sum(np.abs(new_distr - og_distr))) == 0, 'Event labels do not match actual event types'

# Split Testing Set, Training Set, and Validation Set

## Split Training and Testing Sets

Performs a 20-test 20-val 60-train split on all 4-track events, and generates an array of numbers as long as the length of the data to randomize the events. 

In [ ]:
name = ISOTOPE + '_size' + str(sample_size)
all_events = np.load(DATA_PATH + name + '_sampled.npy')

rand_shuffle = np.random.choice(len(all_events), len(all_events), replace = False)


# 20-20 marking for test and validation
test_split = int(len(all_events) * .2)
val_split = int(len(all_events) * .4)

test_data = all_events[rand_shuffle[:test_split],:,:]
val_data = all_events[rand_shuffle[test_split:val_split],:,:]
train_data = all_events[rand_shuffle[val_split:],:,:]

print(test_data.shape, val_data.shape, train_data.shape)
np.save(DATA_PATH + name + 'test', test_data)
np.save(DATA_PATH + name + 'train', train_data)
np.save(DATA_PATH + name + 'val', val_data)

In [ ]:
name = DATA_PATH + ISOTOPE + '_size' + str(sample_size) + '{}.npy'
prev_data = np.load(name.format('_sampled'))
tr = np.load(name.format('train'))
va = np.load(name.format('val'))
te = np.load(name.format('test'))
print(len(prev_data))
print(tr.shape, va.shape, te.shape)
print(len(np.unique(tr[:,:,5])))
print(len(np.unique(va[:,:,12])))
print(len(np.unique(te[:,:,12])))

print(tr.shape, va.shape, te.shape)
print(len(np.unique(tr[:,:,5])))

In [ ]:
assert len(np.unique(np.isnan(tr[:,:,4]))) == 1, 'NaNs in dataset'
assert len(np.unique(np.isnan(va[:,:,4]))) == 1, 'NaNs in dataset'

assert len(np.unique(np.isnan(te[:,:,4]))) == 1, 'NaNs in dataset'
assert np.any(tr[:,:,4]<0) == False, 'Dataset is incorrect, Negative charge values'
assert np.any(va[:,:,4]<0) == False, 'Dataset is incorrect, Negative charge values'

assert np.any(te[:,:,4]<0) == False, 'Dataset is incorrect, Negative charge values'

scaled_val_data = copy.deepcopy(va)
scaled_test_data = copy.deepcopy(te)
scaled_train_data = copy.deepcopy(tr)

print(scaled_test_data.shape, scaled_train_data.shape, scaled_val_data.shape)

In [ ]:
# log scale all charges to reduce large range of values
scaled_test_data[:,:,4] = np.log10(te[:,:,4] + 1e-10)
scaled_train_data[:,:,4] = np.log10(tr[:,:,4] + 1e-10)
scaled_val_data[:,:,4] = np.log10(va[:,:,4] + 1e-10)

In [ ]:
# values correspond to the x,y,z,charge indices
values = [0,1,2,4] 
means_and_stds = []
# standard scaling 
for n in values:
    mean = np.mean(scaled_train_data[:,:,n])
    std = np.std(scaled_train_data[:,:,n])
    means_and_stds.append([mean,std])
    scaled_train_data[:,:,n] = (scaled_train_data[:,:,n] - mean) / std
    scaled_val_data[:,:,n] = (scaled_val_data[:,:,n] - mean) / std
    scaled_test_data[:,:,n] = (scaled_test_data[:,:,n] - mean) / std

name = DATA_PATH + ISOTOPE + '_size' + str(sample_size) + 'scaled_{}'
np.save(name.format('test_data'), scaled_test_data)
    
np.save(name.format('mean_and_std_data'), means_and_stds)    
np.save(name.format('train_data'), scaled_train_data)
np.save(name.format('val_data'), scaled_val_data)

In [ ]:
assert np.sum(np.isnan(scaled_train_data)) == 0, 'NaNs in dataset'
assert np.sum(np.isnan(scaled_val_data)) == 0, 'NaNs in dataset'

assert np.sum(np.isnan(scaled_test_data)) == 0, 'NaNs in dataset'
    
assert np.sum(np.isinf(scaled_train_data)) == 0, 'Infinities in dataset'
assert np.sum(np.isinf(scaled_val_data)) == 0, 'Infinities in dataset'

assert np.sum(np.isinf(scaled_test_data)) == 0, 'Infinities in dataset'

# Get User Desired Inputs and make condensed array 
#### including x/y/z/c values, and then particle ID, event index, and number of tracks

In [ ]:
new_train = np.zeros((len(scaled_train_data), sample_size, len(user_input)), float)
new_val = np.zeros((len(scaled_val_data), sample_size, len(user_input)), float)
new_test = np.zeros((len(scaled_test_data), sample_size, len(user_input)), float)

for i,index in enumerate(user_input):
    new_train[:,:,i] = scaled_train_data[:,:,index]
    new_val[:,:,i] = scaled_val_data[:,:,index]
    new_test[:,:,i] = scaled_test_data[:,:,index]

#new_test[:,:,-1] = scaled_test_data[:,:,6]    #saving the POINT IDs (different from particle IDs)
name = DATA_PATH + ISOTOPE + '_size' + str(sample_size) + '_convert' + PROJECTION + '_{}'

np.save(name.format('train'), new_train)
np.save(name.format('val'), new_val)
np.save(name.format('test'), new_test)

In [ ]:
print(new_train.shape)
print(new_val.shape)
print(new_test.shape)
print(new_val[0,0])
print(new_train[0,0])

assert np.sum(np.isnan(new_train)) == 0, 'NaNs in dataset'
assert np.sum(np.isnan(new_val)) == 0, 'NaNs in dataset'

assert np.sum(np.isnan(new_test)) == 0, 'NaNs in dataset'
    
assert np.sum(np.isinf(new_train)) == 0, 'Infinities in dataset'
assert np.sum(np.isinf(new_val)) == 0, 'Infinities in dataset'

assert np.sum(np.isinf(new_test)) == 0, 'Infinities in dataset'


In [ ]:
# IMPORTANT!!
# check the class distribution before running the num_events_pipeline
unique, counts = np.unique(new_train[:, 0, 6], return_counts=True)
print("Class distribution:")
for cls, count in zip(unique, counts):
    print(f"Class {cls}: {count}")

## Split into features and labels

In [ ]:
scaled_data = DATA_PATH + ISOTOPE + '_size' + str(sample_size) + '_convert' + PROJECTION + '_{}'

train = np.load(scaled_data.format('train')+'.npy')
val = np.load(scaled_data.format('val')+'.npy')
test = np.load(scaled_data.format('test')+'.npy')

train_f = train[:,:,:6]
train_l = train[:,:,6][:,0]

val_f = val[:,:,:6]
val_l = val[:,:,6][:,0]

test_f = test[:,:,:6]
test_l = test[:,:,6][:,0]

np.save(DATA_PATH + ISOTOPE + '_size' + str(sample_size) + '_train_features', train_f)
np.save(DATA_PATH + ISOTOPE + '_size' + str(sample_size) + '_train_labels', train_l)

np.save(DATA_PATH + ISOTOPE + '_size' + str(sample_size) + '_val_features', val_f)
np.save(DATA_PATH + ISOTOPE + '_size' + str(sample_size) + '_val_labels', val_l)

np.save(DATA_PATH + ISOTOPE + '_size' + str(sample_size) + '_test_features', test_f)
np.save(DATA_PATH + ISOTOPE + '_size' + str(sample_size) + '_test_labels', test_l)

In [ ]:
# Make histogram of values (either x,y,z,charge) from selected npy (train, val, test)
# Will need to change PLOT and DATA_SET_NAME to plot X-Y-Z-Q(charge) from training, val, or test
# PLOT = 'Z'
# DATA_SET_NAME = '_train'
# index = PROJECTION.find(PLOT)
# 
# 
# 
# 
#     else:
#         data = np.load(DATA_PATH + ISOTOPE + '_size' + str(sample_size) + '_convert' + PROJECTION + DATA_SET_NAME + '.npy')
#     info = data[:,:,index].flatten()
#     plt.hist(info, density=True, bins=100)
#     plt.ylabel('probibility')
#     plt.xlabel('value')
#     plt.title(PLOT + ' Distributions')
#     plt.show()
#     # plt.savefig(DATA_PATH + '.png', bbox_inches = 'tight') # uncomment to save
# else:
#     print('Value to plot is invalid, change PLOT')